# Titanic survival prediction

This walkthrough follows the same training code as the command-line export. We compare a majority-class baseline with logistic regression and a shallow random forest, select on development folds, and evaluate the selected model on the historical holdout.

The [Kaggle Titanic competition](https://www.kaggle.com/competitions/titanic/data) describes the 891 labeled passenger records used here. Download version 1 of the [checksum-verified Kaggle dataset](https://www.kaggle.com/datasets/yasserh/titanic-dataset) and save its CSV locally as `data/Titanic-Dataset.csv`. You can also set `TITANIC_DATASET` to its absolute path. The notebook displays aggregate statistics and fictional examples.

In [ ]:
import os
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

root = Path.cwd()
if not (root / "train.py").is_file():
    root = root.parent
sys.path.insert(0, str(root))

from app.ml_pipeline import TitanicFeatureEngineer  # noqa: E402
from train import DEFAULT_DATASET, load_dataset, run_experiment, split_data  # noqa: E402

data_path = Path(os.environ.get("TITANIC_DATASET", str(DEFAULT_DATASET)))
data = load_dataset(data_path)
X_train, X_test, y_train, y_test = split_data(data)
print(f"Development rows: {len(X_train)} | Historical holdout rows: {len(X_test)}")

## Development data

The 80/20 stratified split uses seed 42 and preserves the earlier project's row partition for comparability. The old notebook scored its baseline on that holdout before selecting a model and explored outcomes across the entire dataset. Treat the current holdout measurements as a repeat evaluation of known records. The tables below describe the development partition.

In [ ]:
display(y_train.value_counts().rename(index={0: "Did not survive", 1: "Survived"}).to_frame("Rows"))
display(X_train.isna().sum().loc[lambda counts: counts > 0].to_frame("Missing rows"))

## Shared transformations

The transformer turns historical names into five normalized title groups and cabin strings into a recorded-cabin flag. Scenario inputs provide those two fields directly. It also computes family size, a traveling-alone indicator, and `log(1 + fare)`. The indicator lets a linear classifier assign a separate shift to traveling alone. The model excludes names, ticket numbers, passenger IDs, and cabin identifiers.

Median and mode imputation, scaling, and one-hot encoding are fitted independently within each development fold. One-hot encoding ignores categories absent from a fitted fold. Missing age and port use their fitted imputation values. The fictional rows below illustrate the historical CSV and API input formats.

In [ ]:
examples = pd.DataFrame([
    {
        "Pclass": 1, "Sex": "female", "Age": 29, "Embarked": "C",
        "Name": "Example, Mme. Fictional", "Cabin": "F21",
        "SibSp": 1, "Parch": 0, "Fare": 75.0,
    },
    {
        "Pclass": 3, "Sex": "male", "Age": None, "Embarked": "S",
        "Title": "Mr", "HasCabin": False,
        "SibSp": 0, "Parch": 0, "Fare": 8.0,
    },
])
display(TitanicFeatureEngineer().transform(examples))

## Model selection

The majority-class baseline measures the effect of the class imbalance. The logistic model supplies a linear comparison and the shallow forest can represent nonlinear relationships. Five shuffled stratified development folds use the same seed and preprocessing for each candidate. We choose the learned candidate with the highest mean ROC AUC and break an exact tie in favor of logistic regression. Fold standard deviations describe variability across folds.

The published reference run ranks logistic regression first by a small margin. Its mean AUC exceeds the forest's by about 0.0011, much smaller than the fold-to-fold variability. The linear model is easier to inspect and this comparison cannot establish a meaningful advantage. The trainer fits the selected pipeline on the entire development partition before accessing the holdout for evaluation.

In [ ]:
pipeline, report = run_experiment(data)
columns = [
    "model", "accuracy", "precision", "recall", "f1", "roc_auc", "roc_auc_std"
]
display(pd.DataFrame(report["cross_validation"])[columns].round(4))
print(f"Selected: {report['selected_model']}")

## Historical holdout evaluation

ROC AUC measures the ranking of survivors against non-survivors. Precision and recall describe the selected classifier's positive predictions and its coverage of recorded survivors. The confusion matrix counts actual outcomes by row and predicted outcomes by column.

In [ ]:
display(pd.Series({
    metric: report[f"test_{metric}"]
    for metric in ("accuracy", "precision", "recall", "f1", "roc_auc")
}, name="Score").round(4).to_frame())

labels = ["Did not survive", "Survived"]
display(pd.DataFrame(
    report["confusion_matrix"],
    index=pd.Index(labels, name="Actual"),
    columns=pd.Index(labels, name="Predicted"),
))

## Error analysis and limitations

A false negative is a recorded survivor classified as unlikely to survive. A false positive is a recorded non-survivor classified as likely to survive. The group breakdown provides counts and denominators. These slices are small, and class, recorded sex, cabin availability, and fare can reflect social structure and incomplete records.

The split is at passenger level. Known ticket overlap across partitions can make this evaluation optimistic for unseen groups, even though ticket number is excluded from model inputs. It does not establish performance on other manifests, and the score is not calibrated or causal. The modeling population is a subset of people aboard one historical voyage.

In [ ]:
errors = pd.DataFrame(report["error_breakdown"])
display(errors)
for row in errors[errors["field"] == "Sex recorded in dataset"].itertuples():
    print(f"{row.group.capitalize()}: missed {row.false_negatives} of {row.survivors} observed survivors.")
print(f"Holdout records sharing a ticket with development: {report['holdout_rows_sharing_a_train_ticket']}")

## Inference and model export

A fictional profile uses the same fitted transformer as the historical rows. The probability describes the model's estimate under its learned patterns. To regenerate the deployment artifact plus a checksum and package versions, run `python train.py` from the repository root. Notebook execution leaves tracked artifacts untouched.

In [ ]:
scenario = pd.DataFrame([{
    "Pclass": 2, "Sex": "female", "Age": 26, "Embarked": "C",
    "Title": "Miss", "HasCabin": False, "SibSp": 0, "Parch": 0, "Fare": 13.0,
}])
positive_class = list(pipeline.classes_).index(1)
probability = pipeline.predict_proba(scenario)[0, positive_class]
label = "Survived" if pipeline.predict(scenario)[0] == 1 else "Did not survive"
print(f"Model classification: {label}. Estimated probability of survival: {probability:.1%}.")